# Project: Nkwen Traders Branch Performance Dashboard - data preparation
**Goal:** use SQL to build one small, tidy summary table for the dashboard: one row per **month x branch x category**, with revenue, transactions and units, plus the branch's Region, City and Manager.

A summary table is small (well under 200 rows), so your Google Sheet and dashboard stay fast, even on a slow connection.

**What to submit (for this part):** the GitHub commit link to this notebook. The dashboard link, screenshot and manager note go in the same submission (see the activity instructions).

In [ ]:
import sqlite3
import pandas as pd

SALES_URL = "https://innovakickstarter.com/course-files/data-science/data/nkwen_traders_ch2_sales.csv"
BRANCHES_URL = "https://innovakickstarter.com/course-files/data-science/data/nkwen_traders_branches.csv"

# 1. Load the two CSV files into pandas (Colab downloads them, not your phone or laptop)
sales = pd.read_csv(SALES_URL)
branches = pd.read_csv(BRANCHES_URL)

# 2. Create a small SQLite database file and copy both DataFrames into it as tables
conn = sqlite3.connect("nkwen.db")
sales.to_sql("sales", conn, if_exists="replace", index=False)
branches.to_sql("branches", conn, if_exists="replace", index=False)

# 3. A helper: run a SQL query and get the result back as a DataFrame
def run_sql(query):
    return pd.read_sql_query(query, conn)

print("SQLite version:", sqlite3.sqlite_version)
print("Tables:", run_sql("SELECT name FROM sqlite_master WHERE type = 'table'")["name"].tolist())

## Step 1: Build the summary query
Use a CTE or a single query with `JOIN` and `GROUP BY`. Your result must have exactly these columns:

| column | meaning |
|---|---|
| `month_start` | first day of the month as text, for example `2026-03-01` (hint: `strftime('%Y-%m-01', Date)`) |
| `Branch`, `Region`, `City`, `Manager` | from the two tables |
| `Category` | product category |
| `revenue` | sum of TotalSale_FCFA |
| `transactions` | number of sales |
| `units` | sum of Quantity |

Why `month_start` as a full date? Looker Studio recognizes `2026-03-01` as a date, so the date range filter works.

In [ ]:
summary_sql = """
-- TODO: write your summary query here
"""
summary = run_sql(summary_sql)
print(summary.shape)
summary.head(10)

### Self-check 1: does the summary still add up?
A summary must never lose or invent money.

In [ ]:
expected_cols = {"month_start", "Branch", "Region", "City", "Manager", "Category", "revenue", "transactions", "units"}
missing = expected_cols - set(summary.columns)
assert not missing, f"These columns are missing (check your AS names): {missing}"
assert summary["revenue"].sum() == sales["TotalSale_FCFA"].sum(), "Total revenue changed. Check your JOIN and GROUP BY."
assert summary["transactions"].sum() == len(sales), "Total transactions changed. Check your JOIN and COUNT."
assert summary["units"].sum() == sales["Quantity"].sum(), "Total units changed."
print("Success: revenue, transactions and units all match the raw sales table.")

### Self-check 2: is every row complete and one row per group?

In [ ]:
assert summary[["Region", "City", "Manager"]].notna().all().all(), "Some rows have no Region/City/Manager. Check the JOIN condition."
assert summary["month_start"].str.match(r"^\d{4}-\d{2}-01$").all(), "month_start should look like 2026-03-01."
assert not summary.duplicated(subset=["month_start", "Branch", "Category"]).any(), "Some month x branch x category groups appear twice. Check GROUP BY."
print(f"Success: {len(summary)} tidy rows, one per month x branch x category.")

## Step 2: Answer three questions from your summary (not from the raw table)
These numbers go into your note to the operations manager. Use SQL on a new table or pandas on `summary`.
1. Total revenue, total transactions and the **average transaction value** for the whole period.
   Careful: the correct average is `total revenue / total transactions`, **not** the average of the `revenue / transactions` column.
2. Each branch's revenue in its first full month and in June 2026. Which branch is growing fastest?
3. One category pattern that differs between branches (for example the share of Produce or Grains in each branch's revenue).

In [ ]:
# TODO: your calculations for the manager note

## Step 3: Save and download the CSV
The file is tiny (a few kilobytes). Then in Google Sheets: **File > Import > Upload**, choose the file, and pick "Create new spreadsheet".

In [ ]:
summary.to_csv("nkwen_branch_month_summary.csv", index=False)
try:
    from google.colab import files
    files.download("nkwen_branch_month_summary.csv")
except ImportError:
    print("Not running in Colab: the file is saved in your working folder.")

In [ ]:
# Self-check 3: the saved file reads back the same
check = pd.read_csv("nkwen_branch_month_summary.csv")
assert check.shape == summary.shape, "The saved file does not match the summary table."
print("Success: nkwen_branch_month_summary.csv saved with", check.shape[0], "rows.")

### Before you submit
- [ ] **Runtime > Restart session and run all**: every cell runs from top to bottom without an error.
- [ ] Every self-check cell printed its success message.
- [ ] Your answers are written in Markdown cells in your own words, not only as code output.
- [ ] Save to GitHub: **File > Save a copy in GitHub**, choose your repository, write a clear commit message, and click OK.
- [ ] Open the notebook on GitHub, check that the outputs show, and copy the commit link for your submission.